In [ ]:
from Data.preprocessing import DocumentPreprocessor


documentpreprocessor = DocumentPreprocessor()

In [26]:
preprocessor = DocumentPreprocessor()
documents = preprocessor.process_directory()

if documents:
    stats = preprocessor.get_document_stats(documents)
    print("\nDocument Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")


Document Statistics:
  total_documents: 691
  unique_sources: 9
  sources: ['NHS_ThinkKidneys_CKD_Nutritional_Guidelines.pdf', 'UKKA_Commentary_NICE_Hypertension_NG136.pdf', 'UKKA_AKI_Nutrition_Guide.pdf', 'UKKA_Anaemia_CKD_Guideline_Sept2024.pdf', 'Notts_CKD_Management_Guidelines.pdf', 'KDIGO_2024_CKD_Guideline_Draft.pdf', 'KDIGO_2025_Anemia_Guideline_Draft.pdf', 'KDIGO_2025_IgAN_IgAV_Guideline.pdf', 'UKKA_Hyperkalaemia_Management_Guideline.pdf']
  document_types: {'dietary': 28, 'guideline': 663}
  ckd_stage_coverage: {1: 34, 2: 25, 3: 36, 4: 17, 5: 74}
  total_characters: 2072930
  avg_chunk_size: 2999


In [12]:

import fitz


doc = fitz.open("Data/documents/KDIGO_2024_CKD_Guideline_Draft.pdf")
doc.metadata

{'format': 'PDF 1.7',
 'title': 'Topic',
 'author': 'subram1s',
 'subject': '',
 'keywords': '',
 'creator': 'Microsoft® Word for Microsoft 365',
 'producer': 'Microsoft® Word for Microsoft 365',
 'creationDate': "D:20240830124129+01'00'",
 'modDate': "D:20240830094135-04'00'",
 'trapped': '',
 'encryption': None}

In [27]:
from pathlib import Path
from chromadb import logger
from tqdm import tqdm 
pdf_path = Path("Data/documents/KDIGO_2024_CKD_Guideline_Draft.pdf")
pages = []
markdown_content = ""

logger.info(f"Converting {pdf_path.name} with Docling...")
result = self.converter.convert(str(pdf_path))

# Get the full document
doc = result.document

# Extract markdown representation
markdown_content = doc.export_to_markdown()

# Extract plain text
full_text = doc.export_to_text()

# Get title from metadata or filename
title = pdf_path.stem

# Docling doesn't provide page-by-page text easily, so we'll split the text
# into logical pages based on content length (approximate)
# For more accurate page extraction, we'll use the full text and estimate pages
text_lines = full_text.split('\n')

# Estimate pages (roughly 50 lines per page, adjust as needed)
lines_per_page = 50
total_estimated_pages = max(1, len(text_lines) // lines_per_page)

for page_num in range(1, total_estimated_pages + 1):
    start_line = (page_num - 1) * lines_per_page
    end_line = page_num * lines_per_page
    page_text = '\n'.join(text_lines[start_line:end_line])

    if page_text.strip():
        pages.append(
            {
                "text": page_text,
                "page_number": page_num,
                "metadata": {
                    "source": pdf_path.name,
                    "title": title,
                    "page_number": page_num,
                    "total_pages": total_estimated_pages,
                },
            }
        )

# If no pages were created (very short document), create one page with all text
if not pages:
    pages.append(
        {
            "text": full_text,
            "page_number": 1,
            "metadata": {
                "source": pdf_path.name,
                "title": title,
                "page_number": 1,
                "total_pages": 1,
            },
        }
    )

logger.info(f"Extracted {len(pages)} pages from {pdf_path.name}")


NameError: name 'self' is not defined

In [18]:

full_text = "\n\n".join(p["text"] for p in pages)
cleaned_text = documentpreprocessor.clean_text(full_text)

In [24]:
import pprint
pprint.pprint(cleaned_text)

('i KDIGO 2024 CLINICAL PRACTICE GUIDELINE FOR THE MANAGEMENT OF '
 'IMMUNOGLOBULIN A NEPHROPATHY (IgAN) AND IMMUNOGLOBULIN A VASCULITIS (IgAV) '
 'PUBLIC REVIEW DRAFT AUGUST 2024 This is a draft document shared for public '
 'review and feedback only. The content of this draft will change based on the '
 'feedback received, and should not be used for any other purpose beyond its '
 'original intent. ii TABLE OF CONTENTS Tables, figures, and supplementary '
 'materials………………………………………………………………….. ii KDIGO Executive '
 'Committee………………………………………………... vii Reference '
 'keys…………………………………………………………………………... viii Abbreviations and '
 'acronyms………………………………………………………….……………... xi '
 'Notice…………………………………………………………………………………..……………………... xii Work Group membership '
 '………………..………………………………………...………………………... xiii '
 'Abstract……………………………………….……………………………………..…………………...…….. xiv Summary of '
 'recommendation statements and practice points...……………………………………………… 1 '
 'Immunoglobulin A nephropathy…………………………………

In [25]:
pprint.pprint(full_text)

('i \n'
 ' \n'
 ' \n'
 ' \n'
 ' \n'
 ' \n'
 ' \n'
 ' \n'
 'KDIGO 2024 CLINICAL PRACTICE GUIDELINE FOR THE \n'
 'MANAGEMENT OF IMMUNOGLOBULIN A NEPHROPATHY (IgAN) \n'
 'AND IMMUNOGLOBULIN A VASCULITIS (IgAV) \n'
 ' \n'
 'PUBLIC REVIEW DRAFT  \n'
 'AUGUST 2024 \n'
 ' \n'
 ' \n'
 'This is a draft document shared for public review and feedback \n'
 'only. The content of this draft will change based on the feedback \n'
 'received, and should not be used for any other purpose beyond its \n'
 'original intent. \n'
 '\n'
 '\n'
 'ii \n'
 'TABLE OF CONTENTS \n'
 'Tables, figures, and supplementary materials………………………………………………………………….. \n'
 'ii \n'
 'KDIGO Executive '
 'Committee………………………………………………..................................................... \n'
 'vii \n'
 'Reference '
 'keys…………………………………………………………………………..................................... \n'
 'viii \n'
 'Abbreviations and acronyms………………………………………………………….……………................... \n'
 'xi \n'
 'Notice…………………………………………………………………………………..…………………